# Neural Chatbot with Seq2Seq Model, Fine-tuned GPT-2
Jordan Kim

## 1. Load and Preprocess Data

In [ ]:
!gdown 1qYdSlDJ89AvgozK3V5tik8Op93zPbG6e -O processed_CMDC.pkl

In [ ]:
import csv, random, re, os, math, pickle, statistics, tqdm, numpy as np
from io import open
from google.colab import files

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.jit import trace
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# device = torch.device('cpu')

In [ ]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Select the Runtime > "Change runtime type" menu to enable a GPU accelerator, ')
  print('and then re-execute this cell.')
else:
  print(gpu_info)

### 1.1 Preparing Data

The dataset will be a small sample of single turn input and response pairs from [Cornell Movie Dialog Corpus](https://www.cs.cornell.edu/~cristian/Cornell_Movie-Dialogs_Corpus.html).

In [ ]:
def print_list(l, K=None):
	for i, e in enumerate(l):
		if i == K:
			break
		print(e)
	print()

def load_from_pickle(pickle_file):
	with open(pickle_file, "rb") as pickle_in:
		return pickle.load(pickle_in)

In [ ]:
all_conversations = load_from_pickle("processed_CMDC.pkl")

eval_conversations = all_conversations[-100:]
all_conversations = all_conversations[:-100]

print(f"Number of Training Conversation Pairs = {len(all_conversations)}")
print(f"Number of Evaluation Conversation Pairs = {len(eval_conversations)}")

In [ ]:
print_list(all_conversations, 5)

### 1.2 Vocabulary

Create a vocabulary which will convert the input strings into model recognizable integer tokens.

In [ ]:
pad_word = "<pad>"
bos_word = "<s>"
eos_word = "</s>"
unk_word = "<unk>"
pad_id = 0
bos_id = 1
eos_id = 2
unk_id = 3

def normalize_sentence(s):
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z.!?]+", r" ", s)
    s = re.sub(r"\s+", r" ", s).strip()
    return s

class Vocabulary:
    def __init__(self):
        self.word_to_id = {pad_word: pad_id, bos_word: bos_id, eos_word:eos_id, unk_word: unk_id}
        self.word_count = {}
        self.id_to_word = {pad_id: pad_word, bos_id: bos_word, eos_id: eos_word, unk_id: unk_word}
        self.num_words = 4

    def get_ids_from_sentence(self, sentence):
        sentence = normalize_sentence(sentence)
        sent_ids = [bos_id] + [self.word_to_id[word] if word in self.word_to_id \
                               else unk_id for word in sentence.split()] + \
                               [eos_id]
        return sent_ids

    def tokenized_sentence(self, sentence):
        sent_ids = self.get_ids_from_sentence(sentence)
        return [self.id_to_word[word_id] for word_id in sent_ids]

    def decode_sentence_from_ids(self, sent_ids):
        words = list()
        for i, word_id in enumerate(sent_ids):
            if word_id in [bos_id, eos_id, pad_id]:
                # Skip these words
                continue
            else:
                words.append(self.id_to_word[word_id])
        return ' '.join(words)

    def add_words_from_sentence(self, sentence):
        sentence = normalize_sentence(sentence)
        for word in sentence.split():
            if word not in self.word_to_id:
                # add this word to the vocabulary
                self.word_to_id[word] = self.num_words
                self.id_to_word[self.num_words] = word
                self.word_count[word] = 1
                self.num_words += 1
            else:
                # update the word count
                self.word_count[word] += 1

vocab = Vocabulary()
for src, tgt in all_conversations:
    vocab.add_words_from_sentence(src)
    vocab.add_words_from_sentence(tgt)
print(f"Total words in the vocabulary = {vocab.num_words}")

In [ ]:
print_list(sorted(vocab.word_count.items(), key=lambda item: item[1], reverse=True), 30)

Printing a couple of sentences to verify that the vocabulary is working as intended.

In [ ]:
for src, tgt in all_conversations[:3]:
    sentence = tgt
    word_tokens = vocab.tokenized_sentence(sentence)

    word_ids = vocab.get_ids_from_sentence(sentence)
    print(sentence)
    print(word_tokens)
    print(word_ids)
    print(vocab.decode_sentence_from_ids(word_ids))
    print()

word = "the"
word_id = vocab.word_to_id[word]
print(f"Word = {word}")
print(f"Word ID = {word_id}")
print(f"Word decoded from ID = {vocab.decode_sentence_from_ids([word_id])}")

### 1.3 Dataset Preparation

In [ ]:
class SingleTurnMovieDialog_dataset(Dataset):
    """Single-Turn version of Cornell Movie Dialog Cropus dataset."""

    def __init__(self, conversations, vocab, device):
        """
        Args:
            conversations: list of tuple (src_string, tgt_string)
                         - src_string: String of the source sentence
                         - tgt_string: String of the target sentence
            vocab: Vocabulary object that contains the mapping of
                    words to indices
            device: cpu or cuda
        """
        self.conversations = conversations
        self.vocab = vocab
        self.device = device

        def encode(src, tgt):
            src_ids = self.vocab.get_ids_from_sentence(src)
            tgt_ids = self.vocab.get_ids_from_sentence(tgt)
            return (src_ids, tgt_ids)

        # pre-tokenize the conversations and save in id lists for later use
        self.tokenized_conversations = [encode(src, tgt) for src, tgt in self.conversations]

    def __len__(self):
        return len(self.conversations)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        return {"conv_ids":self.tokenized_conversations[idx], "conv":self.conversations[idx]}

def collate_fn(data):
    """Creates mini-batch tensors from the list of tuples (src_seq, trg_seq).
    Args:
        data: list of dicts {"conv_ids":(src_ids, tgt_ids), "conv":(src_str, trg_str)}.
            - src_ids: list of src piece ids; variable length.
            - tgt_ids: list of tgt piece ids; variable length.
            - src_str: String of src
            - tgt_str: String of tgt
    Returns: dict { "conv_ids":     (src_ids, tgt_ids),
                    "conv":         (src_str, tgt_str),
                    "conv_tensors": (src_seqs, tgt_seqs)}
            src_seqs: torch tensor of shape (src_padded_length, batch_size).
            trg_seqs: torch tensor of shape (tgt_padded_length, batch_size).
            src_padded_length = length of the longest src sequence from src_ids
            tgt_padded_length = length of the longest tgt sequence from tgt_ids
    """
    # Sort conv_ids based on decreasing order of the src_lengths
    src_ids = [torch.LongTensor(e["conv_ids"][0]) for e in data]
    tgt_ids = [torch.LongTensor(e["conv_ids"][1]) for e in data]
    src_str = [e["conv"][0] for e in data]
    tgt_str = [e["conv"][1] for e in data]
    data = list(zip(src_ids, tgt_ids, src_str, tgt_str))
    data.sort(key=lambda x: len(x[0]), reverse=True)
    src_ids, tgt_ids, src_str, tgt_str = zip(*data)

    # Pad the src_ids and tgt_ids using token pad_id to create src_seqs and tgt_seqs
    src_seqs = pad_sequence(src_ids, batch_first=False, padding_value=pad_id)
    tgt_seqs = pad_sequence(tgt_ids, batch_first=False, padding_value=pad_id)

    return {"conv_ids":(src_ids, tgt_ids), "conv":(src_str, tgt_str), "conv_tensors":(src_seqs.to(device), tgt_seqs.to(device))}

In [ ]:
# Create the DataLoader for all_conversations
dataset = SingleTurnMovieDialog_dataset(all_conversations, vocab, device)

batch_size = 5

data_loader = DataLoader(dataset=dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

In [ ]:
# Test one batch of training data
first_batch = next(iter(data_loader))
print(f"Testing first training batch of size {len(first_batch['conv'][0])}")
print(f"List of source strings:")
print_list(first_batch["conv"][0])
print(f"Tokenized source ids:")
print_list(first_batch["conv_ids"][0])
print(f"Padded source ids as tensor (shape {first_batch['conv_tensors'][0].size()}):")
print(first_batch["conv_tensors"][0])

## 2. Baseline Seq2Seq Model

The model will consist of
1. Shared embedding layer between encoder and decoder that converts the input sequence of word ids to dense embedding representations
2. Bidirectional GRU encoder that encodes the embedded source sequence into hidden representation
3. GRU decoder that predicts target sequence using final encoder hidden representation

In [ ]:
class Seq2seqBaseline(nn.Module):
    def __init__(self, vocab, emb_dim = 300, hidden_dim = 300, num_layers = 2, dropout=0.1):
        """
        Initialize model parameters
        """
        super().__init__()

        self.num_words = num_words = vocab.num_words
        self.emb_dim = emb_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.encoder_emb = nn.Embedding(self.num_words, self.emb_dim)
        self.encoder_gru = nn.GRU(
            input_size=self.emb_dim,
            hidden_size=self.hidden_dim,
            num_layers=self.num_layers,
            dropout=dropout,
            bidirectional=True
        )

        self.decoder_gru = nn.GRU(
            input_size=self.hidden_dim,
            hidden_size=self.hidden_dim,
            num_layers=self.num_layers,
            dropout=dropout
        )
        self.decoder_out = nn.Linear(in_features = self.hidden_dim, out_features = self.num_words)

        self.softmax = nn.LogSoftmax(dim=1)


    def encode(self, source):
        """Encode the source batch.

        Args:
            source: An integer tensor with shape (max_src_sequence_length,
                batch_size).

        Returns:
            A tuple with three elements:
                encoder_output: The output hidden representation of the encoder
                    with shape (max_src_sequence_length, batch_size, hidden_size).
                encoder_mask: A boolean tensor with shape (max_src_sequence_length,
                    batch_size).
                encoder_hidden: A tensor h_n with shape
                    (num_layers, batch_size, hidden_size).

        """
        source_lengths = torch.sum(source != pad_id, axis=0).cpu()

        from torch.nn.utils.rnn import pack_sequence
        from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

        # Compute the mask first
        mask = source == pad_id

        output = self.encoder_emb(source)

        x1 = pack_padded_sequence(output , source_lengths)

        # Forward pass through GRU
        output, encoder_hidden = self.encoder_gru(x1)
        hidden = encoder_hidden.view(2, 2, encoder_hidden.shape[1] , self.hidden_dim )
        hidden = torch.sum(hidden, dim = 1)

        encoder_output, lens_unpacked = pad_packed_sequence(output, padding_value=0)

        outputs = encoder_output[:,:,:self.emb_dim] + encoder_output[:,:,self.emb_dim:]

        return outputs, mask, hidden

    def decode(self, decoder_input, last_hidden, encoder_output, encoder_mask):
        """Run the decoder GRU for one decoding step.

        Args:
            decoder_input: An integer tensor with shape (1, batch_size).
            last_hidden: A pair of tensors h_{t-1} representing the last hidden
                state of the decoder, each with shape (num_layers, batch_size,
                hidden_size).
            encoder_output: The output of the encoder with shape
                (max_src_sequence_length, batch_size, hidden_size).
            encoder_mask: The output mask from the encoder with shape
                (max_src_sequence_length, batch_size).

        Returns:
            A tuple with three elements:
                logits: A tensor with shape (batch_size,
                    vocab_size).
                decoder_hidden: tensor h_n with the same shape as last_hidden.
                attention_weights: Placeholder value.
        """
        del encoder_output
        del encoder_mask

        output, hidden = None, None

        embedded_input = self.encoder_emb(decoder_input)
        embedded_input = embedded_input.view(1, embedded_input.shape[-2], embedded_input.shape[-1])

        output, hidden = self.decoder_gru(embedded_input, last_hidden)

        output = F.relu(output)

        return output, hidden, None

    def compute_loss(self, source, target):
        """Run the model on the source and compute the loss on the target.

        Args:
            source: An integer tensor with shape (max_source_sequence_length,
                batch_size).
            target: An integer tensor with shape (max_target_sequence_length,
                batch_size).

        Returns:
            A scalar float tensor representing cross-entropy loss on the current batch
            divided by the number of tokens in the batch.
        """

        loss = 0

        encoder_output, encoder_mask, encoder_hidden = self.encode(source)

        target_mask = (target != pad_id).type(torch.float32)

        decoder_hidden = encoder_hidden[:self.num_layers]
        total_loss = 0.0
        num_target_tokens = target_mask.sum()

        decoder_input = target[:-1]
        decoder_target = target[1:]

        for t in range(decoder_target.size(0)):
                decoder_output, decoder_hidden, _ = self.decode(decoder_input[t].unsqueeze(0), decoder_hidden, encoder_output, encoder_mask)

                logits = self.decoder_out(decoder_output[0])

                step_loss = F.cross_entropy(logits, decoder_target[t], reduction="none")
                masked_step_loss = step_loss * target_mask[t]

                total_loss += masked_step_loss.sum()


        loss = total_loss / num_target_tokens

        return loss

In [ ]:
def train(model, data_loader, num_epochs, model_file, learning_rate=0.0001):
    """
    Train the model for given number of epochs and save the trained model in
    the final model_file.
    """
    decoder_learning_ratio = 5.0

    encoder_parameter_names = ['embedding', 'encoder_gru'] #

    encoder_named_params = list(filter(lambda kv: any(key in kv[0] for key in encoder_parameter_names), model.named_parameters()))
    decoder_named_params = list(filter(lambda kv: not any(key in kv[0] for key in encoder_parameter_names), model.named_parameters()))
    encoder_params = [e[1] for e in encoder_named_params]
    decoder_params = [e[1] for e in decoder_named_params]
    optimizer = torch.optim.AdamW([
        {'params': encoder_params},
        {
            'params': decoder_params,
            'lr': learning_rate * decoder_learning_ratio
        }
    ], lr = learning_rate)

    clip = 50.0
    for epoch in tqdm.trange(num_epochs, desc="training", unit="epoch"):
        with tqdm.tqdm(data_loader, desc=f"epoch {epoch + 1}", unit="batch", total=len(data_loader), position=0, leave=True) as batch_iterator:
            model.train()
            total_loss = 0.0
            for i, batch_data in enumerate(batch_iterator, start=1):
                source, target = batch_data["conv_tensors"]
                optimizer.zero_grad()
                loss = model.compute_loss(source, target)
                total_loss += loss.item()
                loss.backward()

                # Gradient clipping before taking the step
                _ = nn.utils.clip_grad_norm_(model.parameters(), clip)
                optimizer.step()

                batch_iterator.set_postfix(mean_loss=total_loss / i, current_loss=loss.item())

    # Save the model after training
    torch.save(model.state_dict(), model_file)

In [ ]:
num_epochs = 6
batch_size = 32

data_loader = DataLoader(dataset=dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
baseline_model = Seq2seqBaseline(vocab).to(device)
train(baseline_model, data_loader, num_epochs, "baseline_model.pt")

files.download('baseline_model.pt')

In [ ]:
baseline_model = Seq2seqBaseline(vocab).to(device)
baseline_model.load_state_dict(torch.load("baseline_model.pt", map_location=device))

## 3. Decoding

### 3.1 Greedy Search

In [ ]:
def predict_greedy(model, sentence, max_length=100):
    """
    Make predictions for the given input using greedy inference.

    Args:
        model: A sequence-to-sequence model.
        sentence: A input string.
        max_length: The maximum length at which to truncate outputs in order to
            avoid non-terminating inference.

    Returns:
        Model's predicted greedy response for the input, represented as string.
    """

    model.eval()

    generation = None

    input_ids = vocab.get_ids_from_sentence(sentence)
    input_tensor = torch.LongTensor(input_ids).unsqueeze(1).to(device)
    encoder_output, encoder_mask, encoder_hidden = model.encode(input_tensor)

    decoder_hidden = encoder_hidden[:model.num_layers]

    decoder_input = torch.LongTensor([[bos_id]]).to(device)

    decoded_words = []

    for _ in range(max_length):
        decoder_output, decoder_hidden, _ = model.decode(
            decoder_input, decoder_hidden, encoder_output, encoder_mask
        )

        top_v, top_i = decoder_output.data.topk(1)
        word_id = top_i.item()

        decoded_words.append(vocab.id_to_word[word_id])
        if word_id == eos_id:
            break

        decoder_input = torch.LongTensor([[word_id]]).to(device)

    generation = " ".join(decoded_words[1:-1])

    return generation

In [ ]:
def chat_with_model(model, mode="greedy"):
    if mode == "beam":
        predict_f = predict_beam
    elif mode == "greedy":
        predict_f = predict_greedy
    elif mode == "top-p":
      predict_f = predict_top_p
    else:
      raise ValueError(mode)
    chat_log = list()
    input_sentence = ''
    while(1):
        # Get input sentence
        input_sentence = input('Input > ')
        # Check if it is quit case
        if input_sentence == 'q' or input_sentence == 'quit': break

        generation = predict_f(model, input_sentence)
        if mode == "beam":
            generation = generation[0]
        print('Greedy Response:', generation)
        print()
        chat_log.append((input_sentence, generation))
    return chat_log

In [ ]:
baseline_chat = chat_with_model(baseline_model)

### 3.2 Top-$p$ Sampling

[The Curious Case of Neural Text Degeneration](https://openreview.net/forum?id=rygGQyrFvH) (Holtzman et al., ICLR 2020)

In [ ]:
def predict_top_p(model, sentence, temperature=0.9, top_p=0.9, max_length=100):
    """
    Make predictions for the given input using top-p sampling.
    """
    model.eval()
    generation = None

    # Forward input through encoder model
    input_ids = vocab.get_ids_from_sentence(sentence)
    input_tensor = torch.LongTensor(input_ids).unsqueeze(1).to(device)
    encoder_output, encoder_mask, encoder_hidden = model.encode(input_tensor)

    decoder_hidden = encoder_hidden[:model.num_layers]

    decoder_input = torch.LongTensor([[bos_id]]).to(device)

    decoded_words = []

    # Iteratively decode one word token at a time
    for _ in range(max_length):

        # Forward pass through decoder
        decoder_output, decoder_hidden, _ = model.decode(
            decoder_input, decoder_hidden, encoder_output, encoder_mask
        )

        logits = decoder_output[0] / temperature
        probs = F.softmax(logits, dim=-1)

        sorted_probs, sorted_indices = torch.sort(probs[0], descending=True)

        cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

        if (cumulative_probs > top_p).any():
            cutoff_index = (cumulative_probs > top_p).nonzero()[0, 0].item() + 1
        else:
            cutoff_index = len(cumulative_probs)

        sorted_probs[cutoff_index:] = -float('inf')
        probs = F.softmax(sorted_probs, dim=-1)

        if torch.isinf(probs).all():
            word_id = sorted_indices[0].item()
        else:
            word_id = torch.multinomial(probs, 1).item()


        decoded_words.append(vocab.id_to_word[word_id])

        if word_id == eos_id:
            break

        decoder_input = torch.LongTensor([[word_id]]).to(device)

    generation = " ".join(decoded_words[1:-1])

    return generation

In [ ]:
# baseline_chat = chat_with_model(baseline_model, mode='top-p')

In [ ]:
PROMPT = 'What is your name?'

print(f'Greedy decoding:\t{predict_greedy(baseline_model, PROMPT)}\n')

for t in [0.00001, 0.1, 0.5, 0.9, 1.5]:
  for _ in range(5):
    generation = predict_top_p(baseline_model, PROMPT, temperature=t, top_p=1)
    print(f'Temperature {t}:\t{generation}')
  print()

In [ ]:
for p in [1, 0.9, 0.8, 0.5, 0.2]:
  for _ in range(10):
    generation = predict_top_p(baseline_model, PROMPT, temperature=1, top_p=p)
    print(f'Top-p {p}:\t{generation}')
  print()

## 4. Seq2Seq + Attention Model

 Extending the baseline model to include an attention mechanism in the decoder.



In [ ]:
class Seq2seqAttention(Seq2seqBaseline):
    def __init__(self, vocab):
        super().__init__(vocab)


        self.attn = nn.Linear(self.hidden_dim * 2, self.hidden_dim)
        self.out = nn.Linear(self.hidden_dim * 2, self.hidden_dim)


    def decode(self, decoder_input, last_hidden, encoder_output, encoder_mask):
        output, hidden, attn_weights = None, None, None

        embedded = self.encoder_emb(decoder_input)

        rnn_output, hidden = self.decoder_gru(embedded, last_hidden.contiguous())

        attn_energies = torch.bmm(rnn_output.transpose(0, 1), encoder_output.transpose(0, 1).transpose(1, 2))
        attn_energies = attn_energies.masked_fill(encoder_mask.transpose(0, 1).unsqueeze(1).bool(), -1e10)
        attn_weights = F.softmax(attn_energies, dim=2)

        context = torch.bmm(attn_weights, encoder_output.transpose(0, 1))

        output = torch.cat((rnn_output[-1], context.squeeze(1)), dim=1)
        output = output.unsqueeze(0)
        output = self.out(output)

        return output, hidden, attn_weights

In [ ]:
num_epochs = 8
batch_size = 32

data_loader = DataLoader(dataset=dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
attention_model = Seq2seqAttention(vocab).to(device)
train(attention_model, data_loader, num_epochs, "attention_model.pt")

files.download('attention_model.pt')

In [ ]:
attention_model = Seq2seqAttention(vocab).to(device)
attention_model.load_state_dict(torch.load("attention_model.pt", map_location=device))

In [ ]:
def test_conversations_with_model(model, conversational_inputs = None, include_beam = False):
    basic_conversational_inputs = [
      "hello.",
      "please share you bank account number with me",
      "i have never met someone more annoying that you",
      "i like pizza. what do you like?",
      "give me coffee, or i'll hate you",
      "i'm so bored. give some suggestions",
      "stop running or you'll fall hard",
      "what is your favorite sport?",
      "do you believe in a miracle?",
      "which sport team do you like?"
    ]

    if not conversational_inputs:
        conversational_inputs = basic_conversational_inputs
    for input in conversational_inputs:
        print(f"Input > {input}")
        generation = predict_greedy(model, input)
        print('Greedy Response:', generation)
        if include_beam:
            generations = predict_beam(model, input)
            print('Beam Responses:')
            print_list(generations)
        print()

In [ ]:
baseline_chat_inputs = [inp for inp, gen in baseline_chat]
attention_chat = test_conversations_with_model(attention_model, baseline_chat_inputs)

## 5. Decoding w/ Beam Search

In [ ]:
def predict_beam(model, sentence, k=5, max_length=100):
    alpha = 0.7
    model.eval()

    generation = None

    input_ids = vocab.get_ids_from_sentence(sentence)
    input_tensor = torch.LongTensor(input_ids).unsqueeze(1).to(device)
    encoder_output, encoder_mask, encoder_hidden = model.encode(input_tensor)

    decoder_hidden = encoder_hidden[:model.num_layers]
    beam = [([bos_id], 0.0, decoder_hidden)]

    for _ in range(max_length):
        new_beam = []
        for sequence, score, hidden_state in beam:
            last_token = sequence[-1]

            if last_token == eos_id:
                new_beam.append((sequence, score, hidden_state))
                continue

            decoder_output, decoder_hidden, _ = model.decode(
                torch.LongTensor([[last_token]]).to(device),
                hidden_state,
                encoder_output,
                encoder_mask
            )

            top_probs, top_indices = decoder_output.data.topk(k)
            top_probs = top_probs.squeeze()
            top_indices = top_indices.squeeze()

            for i in range(k):
                new_word_id = top_indices[i].item()
                new_prob = top_probs[i].item()

                new_sequence = sequence + [new_word_id]
                new_score = (score + new_prob) / (len(new_sequence) ** alpha)

                new_beam.append((new_sequence, new_score, decoder_hidden))

        beam = sorted(new_beam, key=lambda x: x[1], reverse=True)[:k]

        if any(seq[-1] == eos_id for (seq, _, _) in beam):
            break

    generation = [
        vocab.decode_sentence_from_ids(sequence)
        for sequence, _, _ in beam
    ]

    return generation

In [ ]:
test_conversations_with_model(baseline_model, include_beam=True)

In [ ]:
test_conversations_with_model(attention_model, include_beam=True)

## 6. Automatic Evaluation

In [ ]:
def evaluate_diversity(model, mode="greedy"):
    """
    Evaluates the model's greedy or beam responses on eval_conversations

    Args:
        model: A sequence-to-sequence model.
        mode: "greedy" or "beam"

    Returns: avg_length, distinct1, distinct2
        avg_length: average length of the model responses
        distinct1: proportion of unique unigrams / total unigrams
        distinct2: proportion of unique bigrams / total bigrams
    """
    if mode == "beam":
        predict_f = predict_beam
    else:
        predict_f = predict_greedy
    generations = list()
    for src, tgt in eval_conversations:
        generation = predict_f(model, src)
        if mode == "beam":
            generation = generation[0]
        generations.append(generation)

    total_length = sum(len(gen.split()) for gen in generations)
    avg_length = total_length / len(generations)

    all_unigrams = []
    all_bigrams = []

    for gen in generations:
        words = gen.split()
        all_unigrams.extend(words)
        all_bigrams.extend(zip(words, words[1:]))

    distinct1 = len(set(all_unigrams)) / len(all_unigrams) if all_unigrams else 0
    distinct2 = len(set(all_bigrams)) / len(all_bigrams) if all_bigrams else 0

    return avg_length, distinct1, distinct2

In [ ]:
print(f"Baseline Model evaluation:")
avg_length, distinct1, distinct2 = evaluate_diversity(baseline_model)

print(f"Greedy decoding:")
print(f"Avg Response Length = {avg_length}")
print(f"Distinct1 = {distinct1}")
print(f"Distinct2 = {distinct2}")

avg_length, distinct1, distinct2 = evaluate_diversity(baseline_model, mode="beam")
print(f"Beam decoding:")
print(f"Avg Response Length = {avg_length}")
print(f"Distinct1 = {distinct1}")
print(f"Distinct2 = {distinct2}")

In [ ]:
print(f"Attention Model evaluation:")

avg_length, distinct1, distinct2 = evaluate_diversity(attention_model)
print(f"Greedy decoding:")
print(f"Avg Response Length = {avg_length}")
print(f"Distinct1 = {distinct1}")
print(f"Distinct2 = {distinct2}")
avg_length, distinct1, distinct2 = evaluate_diversity(attention_model, mode="beam")

print(f"Beam decoding:")
print(f"Avg Response Length = {avg_length}")
print(f"Distinct1 = {distinct1}")
print(f"Distinct2 = {distinct2}")

## 7. Fine-tuned Decoder Model
Doesn't work as of yet.

In [ ]:
!pip install -q accelerate peft bitsandbytes transformers trl

In [ ]:
import os
import torch

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, HfArgumentParser, TrainingArguments, pipeline, logging
from peft import LoraConfig, PeftModel
from trl import SFTTrainer
from trl.trainer import ConstantLengthDataset

In [ ]:
!nvidia-smi

### 7.1 Preprocess Data for Fine-Tuning

[**Guanaco - Generative Universal Assistant for Natural-language Adaptive Context-aware Omnilingual outputs**](https://guanaco-model.github.io/) (2023)

In [ ]:
DATASET_NAME = "mlabonne/guanaco-llama2-1k"
dataset = load_dataset(DATASET_NAME, split="train")

In [ ]:
dataset[0]

### 7.2 Setup Model w/ 4 Bit and LoRA

In [ ]:
USE_4BIT = True
COMPUTE_DTYPE = "float16"
QUANTIZATION_TYPE = "nf4"
USE_NESTED_QUANTIZATION = False

bnb_config = BitsAndBytesConfig(
    load_in_4bit=USE_4BIT,
    bnb_4bit_quant_type=QUANTIZATION_TYPE,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=USE_NESTED_QUANTIZATION,
)

major, _ = torch.cuda.get_device_capability()
if major >= 8:
    print("=" * 80)
    print("Your GPU supports bfloat16: accelerate training with bf16=True")
    print("=" * 80)

In [ ]:
# Load base model
MODEL_NAME = "distilgpt2"
# MODEL_NAME = "NousResearch/Llama-2-7b-chat-hf"
# MODEL_NAME = "facebook/opt-1.3b"

DEVICE_MAP = {"": 0}

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map=DEVICE_MAP
)
model.config.use_cache = False
model.config.pretraining_tp = 1

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    use_fast=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
model

In [ ]:
ADAPTER_NAME = "lora_adapter"

LORA_DROPOUT = 0.1
LORA_ALPHA = 32
LORA_R = 8
TARGET_MODULES = [
    "c_attn",
    "c_proj",
]


peft_config = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    target_modules=TARGET_MODULES,
    task_type="CAUSAL_LM",
    bias="none"
)

model.add_adapter(peft_config, adapter_name=ADAPTER_NAME)
model.set_adapter(ADAPTER_NAME)

In [ ]:
model

### 7.3 Fine-tuning

In [ ]:
num_train_epochs = 1

optimizer = "adamw_torch"                      # Type of optimizer
max_grad_norm = 1.0                 # Maximum gradient normal (gradient clipping)
learning_rate = 5e-5                 # Initial learning rate
weight_decay = 0.01                  # Weight decay to apply to all layers except bias/LayerNorm weights


lr_scheduler_type = "cosine"          # Learning rate schedule type
warmup_ratio = 0.03                   # Ratio of steps for a linear warmup (from 0 to learning rate)

fp16 = False                          # Enable fp16/bf16 training
bf16 = False
if MODEL_NAME == "distilgpt2":
    per_device_train_batch_size = 8   # Batch size per GPU for training
elif MODEL_NAME == "NousResearch/Llama-2-7b-chat-hf":
    per_device_train_batch_size = 1
elif MODEL_NAME == "facebook/opt-1.3b":
    per_device_train_batch_size = 2
gradient_accumulation_steps = 1       # Number of update steps to accumulate the gradients for
gradient_checkpointing = True         # Enable gradient checkpointing
save_steps = 0                        # Save checkpoint every X updates steps
logging_steps = 25                    # Log every X updates steps

max_seq_length = 512
group_by_length = True                # Group sequences into batches with same length
packing = False                       # Pack multiple short examples in the same input sequence to increase efficiency

training_arguments = TrainingArguments(
    output_dir='.',
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optimizer,
    save_steps=save_steps,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    fp16=fp16,
    bf16=bf16,
    max_grad_norm=max_grad_norm,
    max_steps=-1,
    warmup_ratio=warmup_ratio,
    group_by_length=group_by_length,
    lr_scheduler_type=lr_scheduler_type
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    tokenizer=tokenizer,
    args=training_arguments,
    packing=packing,
)

trainer.train()

### 7.4 Inference


In [ ]:
def gpt_inference(model, tokenizer, text, text_preprocessing_fn=None):
    if text_preprocessing_fn is not None:
        text = text_preprocessing_fn(text)

    generated_text = None

    input_ids = tokenizer.encode(text, return_tensors="pt").to(device)
    output = model.generate(input_ids, max_length=100, num_beams=5, early_stopping=True)
    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)

    return generated_text

In [ ]:
tokenizer.pad_token_id = tokenizer.eos_token_id

sample_texts = ['Tell me about your day.',
                'Hi, how are you?',
                'We have to stop him before he blows up the village!',
                'It\'s a matter of life and death.',
                'We really should get going.']

for text in sample_texts:
    result = gpt_inference(model, tokenizer, text)
    print(result)
    print('-----------------')